# Этап 4. Визуализация электропотребления

Данные берутся из `data/raw` внутри клона GitHub. Профили считаются по исходным минутным значениям, а не из повторно загружаемых ZIP предыдущих этапов.

## Имена переменных

- `source` — выбранный исходный файл; `measurements` — прочитанные измерения в `pandas`.
- `minutes` — равномерная минутная сетка; `month_ids` — коды выбранных месяцев в формате `YYYY-MM`.
- `profile_table` — по 1440 минут суточного профиля каждого месяца.
- `minute_of_day` — номер минуты суток от 0 (00:00) до 1439 (23:59).
- `mean_kw` — средняя мощность по дням для этой минуты суток, в кВт.
- `n_observed` — число дней, для которых измерение этой минуты известно.
- `n_expected` — число ожидаемых измерений этой минуты внутри исследуемого периода.
- `coverage_pct` — полнота этой минуты: `100 × n_observed / n_expected`, в процентах.
- `labplot_table` — таблица для LabPlot: `Hour` — время суток в часах; `P_YYYY-MM_kW` — средняя мощность месяца; `Coverage_YYYY-MM_pct` — полнота месяца.

Месяцы 2008 года используются, если они есть в источнике. Для другого файла автоматически берутся первые доступные месяцы; называть их всеми сезонами года без проверки **нельзя**.

In [ ]:
# Одна настройка пути к репозиторию. Исходные файлы не загружаются вручную.
from pathlib import Path
import subprocess
import sys

repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ipd_analysis.py").is_file()), None)
if repo_dir is None:
    repo_dir = Path("/content/ipd")
    if not (repo_dir / "ipd_analysis.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sonyxoxo/ipd.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

from ipd_analysis import choose_source, load_measurements, regular_minutes, result_folder, save_csv

# Если в data/raw несколько файлов, задайте относительный путь явно:
# source = choose_source("data/raw/household_power_consumption.zip")
source = choose_source()
measurements = load_measurements(source)  # Один pd.read_csv на весь ноутбук.
output_dir = result_folder(source)
output_dir.mkdir(parents=True, exist_ok=True)
print("Источник:", source)
print("Строк:", len(measurements))

## Расчёт суточных профилей и файл LabPlot

`select_months()` выбирает существующие месяцы. `daily_profiles()` создаёт 1440 минут каждого выбранного месяца с правильным учётом отсутствующих строк. `export_labplot()` формирует широкий формат, который удобно импортировать в LabPlot.

**Важно:** если какие-то минуты вообще не наблюдались, соответствующее среднее будет `NaN`, а не нулём.

In [ ]:
from ipd_analysis import select_months, daily_profiles, export_labplot

minutes = regular_minutes(measurements)
month_ids = select_months(minutes)
profile_table = daily_profiles(minutes, month_ids)
labplot_table = export_labplot(profile_table)
display(profile_table.head())
save_csv(profile_table, output_dir / "04_daily_profiles.csv")
save_csv(labplot_table, output_dir / "04_labplot_profiles.csv")
print("Месяцы:", month_ids)

## Графики мощности и полноты

- `chart_path` — путь к изображению PNG в папке результатов этого исходного файла.
- На верхнем графике — средняя активная мощность, **кВт**; на нижнем — полнота, **%**. Различные единицы не смешиваются.
- `daily_chart_path` — график одного фактически наблюдавшегося дня.
- `missing_chart_path` — график недели вокруг самой длинной серии пропусков.
- `month_chart_path` — сравнение двух месяцев, включая различие в полноте наблюдений.
- Графики сохраняются в папку репозитория `результаты/…`, а не в архивы из Colab.

In [ ]:
from run_analysis import plot_profiles, plot_one_day, plot_missing_week, plot_month_comparison

chart_path = output_dir / "04_daily_profiles.png"
daily_chart_path = output_dir / "04_one_day.png"
missing_chart_path = output_dir / "04_missing_week.png"
month_chart_path = output_dir / "04_month_comparison.png"
plot_profiles(profile_table, chart_path)
plot_one_day(minutes, daily_chart_path)
plot_missing_week(minutes, missing_chart_path)
plot_month_comparison(minutes, month_chart_path)
print("Графики сохранены в:", output_dir)
from IPython.display import Image, display
display(Image(filename=str(chart_path)))